# Stan 1D Gaussian Random Restarts

Runs Stan ADVI random restarts using prefix-length reruns so final variational draws provide trajectory checkpoints.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/1dgaussian_knownvar/random_restarts")
STAN_FILE_NAME = "stan_1dgaussian_knownvar.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


In [ ]:
from modulars import gaussian_1d, gaussian_1d_posterior

mu_prior, sigma_prior = 0.0, 10.0
sigma_like = 1.0
n_samples = 100
mu_like = 3.0

data = gaussian_1d(n_samples, mu_like, sigma_like, seed=0).astype(float)
true_mu_post, true_sigma_post = gaussian_1d_posterior(
    data,
    mu_prior=mu_prior,
    sigma_prior=sigma_prior,
    sigma_like=sigma_like,
)

stan_data = {
    "N": len(data),
    "y": data,
    "sigma_like": sigma_like,
    "mu_prior": mu_prior,
    "sigma_prior": sigma_prior,
}
PARAM_COLUMNS = ["mu"]
LOG_COLUMNS = []
true_mu_post, true_sigma_post


In [ ]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 50, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "stan_cmdstan_output"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


In [ ]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_tuple

stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=results_dir / "cmdstan_runs",
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
)

single_means, single_stds, multi_means, multi_stds = stan_result_tuple(stan_result)
iterations = stan_result["iterations"]
print(single_means.shape, single_stds.shape, multi_means.shape, multi_stds.shape)
print(f"tracked iterations: {iterations[:5]} ... {iterations[-5:]}")
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")


In [ ]:
from modulars import save_rr_tracking_csv

TRACKING_CSV = Path(os.environ.get("SIMPLEVI_STAN_TRACKING_DIR", "processed_tracking")) / "rr_stan_tracking.csv"
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
    iterations=iterations,
)
TRACKING_CSV


In [ ]:
from modulars import load_rr_tracking_csv
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d

TRACKING_CSV = Path(os.environ.get("SIMPLEVI_STAN_TRACKING_DIR", "processed_tracking")) / "rr_stan_tracking.csv"
single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)

best_mu, best_std = (true_mu_post, true_sigma_post)
plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r"$\mu$", label_prefix="Stan "
)
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r"$\mu$", 1, label_prefix="Stan "
)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r"$\mu$", 100, label_prefix="Stan "
)
